# Coursework: Baseline Performance Analysis (Option 4)
This notebook performs the **Control Experiment** for the Robotic Vision Robustness study.
    
**Objective:** To establish a mathematically optimal baseline model trained strictly on clean, "simulation-grade" data. This model serves as the comparison point to measure the "Sim-to-Real" gap when exposed to sensor degradation.

**Methodology:**
1. Hyperparameter search using **Optuna** (50 trials).
2. Optimization target: **Clean Validation Accuracy**.
3. Multi-seed averaging (4 seeds) for scientific rigor.
4. Evaluation across 4 levels of Gaussian noise to measure catastrophic drop-off.

In [ ]:
# If running on Google Colab, uncomment the line below to install the necessary Optuna packages:
#!pip install optuna optuna-integration[tfkeras]
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import pandas as pd
import matplotlib.pyplot as plt

# Create outputs directory
os.makedirs('outputs', exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
print("Loading CIFAR-100 dataset...")
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode='fine')
x_train, x_test = x_train / 255.0, x_test / 255.0

def add_gaussian_noise(images, severity=0.1):
    noise = tf.random.normal(shape=tf.shape(images), mean=0.0, stddev=severity, dtype=tf.float64)
    noisy_images = images + noise
    return tf.clip_by_value(noisy_images, 0.0, 1.0).numpy()

print("Generating Test Sets for Sim-to-Real Analysis...")
x_test_noisy_low = add_gaussian_noise(x_test, severity=0.1)
x_test_noisy_med = add_gaussian_noise(x_test, severity=0.25)
x_test_noisy_high = add_gaussian_noise(x_test, severity=0.4)

BATCH_SIZE = 64
base_train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000)
val_dataset_clean = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_low = tf.data.Dataset.from_tensor_slices((x_test_noisy_low, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_med = tf.data.Dataset.from_tensor_slices((x_test_noisy_med, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset_high = tf.data.Dataset.from_tensor_slices((x_test_noisy_high, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_model(conv_blocks, filters, dropout_rate):
    inputs = keras.Input(shape=(32, 32, 3))
    x = inputs
    for i in range(conv_blocks):
        x = layers.Conv2D(filters * (2**i), (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters * (2**i), (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = layers.Dropout(dropout_rate)(x)
        
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(100, activation='softmax')(x)
    return keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
# Configuration
EPOCHS = 10 # Short epochs for the notebook demonstration; use 40+ for final research
N_TRIALS = 3
SEEDS = [42, 123, 999]

def objective(trial):
    conv_blocks = trial.suggest_int("conv_blocks", 2, 4)
    filters = trial.suggest_categorical("filters", [32, 64])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    
    seed_accuracies = []
    pruning_callback = optuna.integration.TFKerasPruningCallback(trial, "val_accuracy")
    
    for i, seed in enumerate(SEEDS):
        print(f"  -> Training with seed {seed}...")
        keras.utils.set_random_seed(seed)
        
        model = build_model(conv_blocks, filters, dropout_rate)
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
        model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        
        def augment(image, label):
            image = tf.image.random_flip_left_right(image) # Basic structural augmentation ONLY
            return image, label
            
        trial_dataset = base_train_dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        trial_dataset = trial_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
        callbacks = [pruning_callback] if i == 0 else []
        
        history = model.fit(
            trial_dataset,
            epochs=EPOCHS,
            validation_data=val_dataset_clean, # TARGET: MASTERING CLEAN DATA
            verbose=0,
            callbacks=callbacks
        )
        seed_accuracies.append(history.history["val_accuracy"][-1])
        
    trial.set_user_attr("std_dev", float(np.std(seed_accuracies)))
    trial.set_user_attr("seed_scores", str([float(acc) for acc in seed_accuracies]))
    return np.mean(seed_accuracies)

In [ ]:
print("Starting Bayesian Optimization for Baseline Model...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("\nBest Hyperparameters:")
print(study.best_params)

# Save results
study.trials_dataframe().to_csv('outputs/baseline_optuna_results.csv', index=False)

In [ ]:
best_params = study.best_params
print("Training final optimal Baseline model...")
keras.utils.set_random_seed(42)

final_model = build_model(
    conv_blocks=best_params["conv_blocks"],
    filters=best_params["filters"],
    dropout_rate=best_params["dropout_rate"]
)

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_params["learning_rate"], weight_decay=best_params.get("weight_decay", 0.0)), 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)

final_batch_size = best_params.get("batch_size", 64)
final_train_dataset = base_train_dataset.map(lambda x, y: (tf.image.random_flip_left_right(x), y)).batch(final_batch_size).prefetch(tf.data.AUTOTUNE)

final_model.fit(final_train_dataset, epochs=20, validation_data=val_dataset_clean, verbose=1)

print("\nEvaluating Baseline across noise levels...")
acc_clean = final_model.evaluate(val_dataset_clean, verbose=0)[1]
acc_low = final_model.evaluate(val_dataset_low, verbose=0)[1]
acc_med = final_model.evaluate(val_dataset_med, verbose=0)[1]
acc_high = final_model.evaluate(val_dataset_high, verbose=0)[1]

plt.figure(figsize=(8,5))
plt.plot(['Clean', 'Low', 'Med', 'High'], [acc_clean, acc_low, acc_med, acc_high], marker='o', color='r')
plt.title('Baseline Model Performance Drop-off')
plt.ylabel('Accuracy')
plt.grid(True)
plt.savefig('outputs/baseline_final_dropoff.png')
plt.show()